# 03 · Integration tutorial — Claire × Shilin

This eight-part tutorial joins Claire's [fixed on-chain release](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de) with Shilin's v1 linkage and prespecified v2 evidence extension. Claire's source and files are unchanged. The unit of the final table is a committed, decoded creation event. The five-minute retrospective cohort is a bounded pilot; independent joint review remains pending.

[Protocol](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/EXTENSION_PROTOCOL.md) · [Dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/DATA_DICTIONARY.md) · [Validation report](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/VALIDATION_REPORT.md) · [V1 tables](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v1/release) · [V2 tables](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/release). Run code cells in order in a CPU runtime.

<a id="part-1"></a>
## Part 1 — Question and navigation

How do exact chain records connect to later metadata observations while retaining failures and uncertainty? The event declares a metadata URI; a later HTTP response supplies field values and retrieval time. The join does not establish destination ownership or launch-time availability.

1. [Question](#part-1) · 2. [Sources](#part-2) · 3. [Inputs](#part-3) · 4. [Compatibility](#part-4) · 5. [Join](#part-5) · 6. [Examples](#part-6) · 7. [Coverage](#part-7) · 8. [Validation](#part-8).

<a id="part-2"></a>
## Part 2 — Sources and join contract

Claire's fixed revision covers 2026-09-14 12:00–12:05 UTC across three chains. Shilin's [v1 selection](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v1/build_cohort.py) contains 116 committed creation events: 61 Pump.fun, 55 Four.meme, and zero recognized Base Clanker creations. The extension reuses exactly these 116 events. `launch_record_id` is the event key; `object_id` is chain-qualified. Metadata and images have distinct URI/request keys. Event time and retrieval time remain separate; no fuzzy name or symbol join is used.

```mermaid
flowchart LR
 A[Claire creation event] --> B[Exact declared metadata URI]
 B --> C[Shilin v1 response and URL fields]
 C --> D[Shilin v2 field and image evidence]
 A --> E[116-event coverage ledger]
 C --> E
 D --> E
```

In [ ]:
import os, sys, json, tarfile, tempfile, urllib.request
from pathlib import Path
from collections import Counter
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow>=23,<26"])
    import pyarrow as pa
    import pyarrow.parquet as pq

COMMIT = "0655881"
local = os.environ.get("PILOT_LOCAL_REPO")
if local:
    ROOT = Path(local).expanduser().resolve()
    print("Using local author checkout:", ROOT)
else:
    url = f"https://codeload.github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/tar.gz/{COMMIT}"
    request = urllib.request.Request(url, headers={"User-Agent": "shilin-offchain-v2-colab/1.0"})
    body = urllib.request.urlopen(request, timeout=90).read()
    temp = Path(tempfile.mkdtemp(prefix="shilin-v2-"))
    archive = temp / "input.tar.gz"
    archive.write_bytes(body)
    with tarfile.open(archive, "r:gz") as tar:
        prefix = tar.getnames()[0].split("/")[0]
        tar.extractall(temp, filter="data")
    ROOT = temp / prefix
V1 = ROOT / "pilots/shilin-offchain-v1/release"
V2 = ROOT / "pilots/shilin-offchain-v2/release"
sys.path.insert(0, str(ROOT / "pilots/shilin-offchain-v2"))
from verify_extension import verify
result = verify(V2, V1)
assert result["passed"], result
print("Fixed public release verified; PyArrow", pa.__version__)

<a id="part-3"></a>
## Part 3 — Acquire pinned public inputs

The setup downloads a pinned repository commit, then checks the v1 and v2 release manifests. It reads released Parquet tables only; third-party JSON and image bodies have not been redistributed. See the [acquisition protocol](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/EXTENSION_PROTOCOL.md) for the complete request rules, fixed denominators, and 2 MiB image cap.

In [ ]:
from verify_extension import verify as verify_v2
sys.path.insert(0, str(ROOT / "pilots/shilin-offchain-v1"))
from verify_release import verify as verify_v1
v1_result=verify_v1(V1)
v2_result=verify_v2(V2,V1)
assert v1_result["passed"] and v2_result["passed"]
cohort=pq.read_table(V1/"onchain_launch_cohort.parquet").to_pylist()
base_coverage=pq.read_table(V1/"coverage_ledger.parquet").to_pylist()
ext_coverage=pq.read_table(V2/"event_extension_coverage.parquet").to_pylist()
print("Verified public releases; event counts:",len(cohort),len(base_coverage),len(ext_coverage))

<a id="part-4"></a>
## Part 4 — Inspect keys and denominators

An event can share a metadata URI with another event. URI-level request deduplication must therefore leave the event denominator intact. The v1 coverage states distinguish a successful JSON without a covered URL field from a source that returned 403 or was not attempted.

In [ ]:
event_ids={r["launch_record_id"] for r in cohort}
assert len(cohort)==len(event_ids)==116
assert {r["launch_record_id"] for r in base_coverage}==event_ids
assert {r["launch_record_id"] for r in ext_coverage}==event_ids
assert len({r["metadata_uri_declared"] for r in cohort if r["metadata_uri_declared"]})==57
print("Platforms:",dict(Counter(r["platform_id"] for r in cohort)))
print("V1 coverage:",dict(Counter(r["coverage_state"] for r in base_coverage)))

<a id="part-5"></a>
## Part 5 — Join without losing unknowns

The event-level join uses `launch_record_id`. V1 has one coverage outcome per event, and v2 has one extension outcome per event. The left join preserves 116 rows; image requests and URL declarations stay in their own many-to-one tables rather than multiplying event counts.

In [ ]:
v1_by_id={r["launch_record_id"]:r for r in base_coverage}
v2_by_id={r["launch_record_id"]:r for r in ext_coverage}
joined=[{**r,"v1_state":v1_by_id[r["launch_record_id"]]["coverage_state"],
         "v2_metadata_state":v2_by_id[r["launch_record_id"]]["metadata_refetch_state"],
         "v2_image_state":v2_by_id[r["launch_record_id"]]["image_acquisition_state"]} for r in cohort]
assert len(joined)==len({r["launch_record_id"] for r in joined})==116
print("Joined states:",dict(Counter(r["v1_state"] for r in joined)))

<a id="part-6"></a>
## Part 6 — Inspect positive and unresolved evidence

V1 reports 39 URL field declarations on 29 Pump events. V2 adds 16 `coin_community` URL declarations and image-request outcomes; these are separate field observations. The original 39 URL values received deterministic target-category flags. A post URL is not an account, a website-field social URL needs review, and a 403 is access restriction here rather than evidence of absence.

In [ ]:
decl=pq.read_table(V1/"offchain_declarations.parquet").to_pylist()
field=pq.read_table(V2/"metadata_field_audit.parquet").to_pylist()
checks=pq.read_table(V2/"event_metadata_checks.parquet").to_pylist()
sem=pq.read_table(V2/"url_semantic_checks.parquet").to_pylist()
community=[r for r in field if r["field_name"]=="coin_community" and r["field_state"]=="nonempty"]
print("V1 URL declarations:",len(decl),"V2 community values:",len(community))
print("Literal name/symbol matches:",sum(r["name_exact_match"] is True for r in checks),sum(r["symbol_exact_match"] is True for r in checks))
print("Target-category flags:",dict(Counter(r["semantic_state"] for r in sem)))
assert len(decl)==len(sem)==39 and len(community)==16 and len(checks)==61

<a id="part-7"></a>
## Part 7 — Describe coverage with the event denominator

This descriptive table reports observable source states in the five-minute cohort. It is not an estimate of project quality, token outcomes, or social-account ownership. The v2 image state can cover multiple events sharing one URI, so resource-level and event-level counts have different denominators.

In [ ]:
for platform in sorted({r["platform_id"] for r in joined}):
    rows=[r for r in joined if r["platform_id"]==platform]
    print(platform,{"events":len(rows),"v1":dict(Counter(r["v1_state"] for r in rows)),
                    "v2_images":dict(Counter(r["v2_image_state"] for r in rows))})
images=pq.read_table(V2/"image_acquisition.parquet").to_pylist()
print("Distinct image URI requests:",len(images),"saved:",sum(r["status"]=="response_saved" for r in images))
assert len(images)==56

<a id="part-8"></a>
## Part 8 — Validation and interpretation

The [measured report](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/0655881/pilots/shilin-offchain-v2/VALIDATION_REPORT.md) documents 57/57 stable metadata hashes, 798 audited field rows, 54/56 image bodies within the fixed cap, and 48 events queued for independent review. Thirty single-block CIDv1 raw-image hashes matched; other CID forms were not verified by that method. Four.meme contributes no positive off-chain link because exact-address access was restricted. The queue has no completed independent reviews. This retrospective extension cannot establish creation-time URL availability; a formal Data Descriptor needs a separately frozen prospective cohort, rights clearance, and adjudication.

In [ ]:
final=verify_v2(V2,V1)
queue=pq.read_table(V2/"human_review_queue.parquet").to_pylist()
assert final["passed"] and len(queue)==48
assert all(r["review_status"]=="pending_independent_human_review" for r in queue)
print("Public checks passed:",sum(final["checks"].values()),"/",len(final["checks"]))
print("Independent reviews completed: 0; joint handoff pending.")